# Preferred nutrients

Adds one column per nutrient to `recipe_user_ratings_prices.csv`: the user's preferred value of it, a weighted
average over the recipes they reviewed in which a review with a lower rating gets less weight:
weight = 1 / 2^(5 − rating), so 5 → 1, 4 → 1/2, 3 → 1/4, 2 → 1/8, 1 → 1/16 and 0 → 1/32.

- **Rounded.** Each weighted average is rounded to the nearest integer, a half going up.

The output, `recipe_user_ratings_preferences_prices.csv`, is the input of
[model_training_reranker_avgemb.ipynb](model_training_reranker_avgemb.ipynb).

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
INPUT_DIR = "./input_stage2"
# The training notebook's nutrient columns, in its order: UserEncoder averages them with these preferences column by
# column.
NUTRIENT_COLUMNS = ["calories (g)", "total fat (g)", "sugar (g)", "sodium (g)", "protein (g)",
                    "saturated fat (g)", "carbohydrates (g)"]
PREFERENCE_COLUMNS = [f"preferred {column}" for column in NUTRIENT_COLUMNS]

recipe_user_interaction_df = pd.read_csv(INPUT_DIR+"/recipe_user_ratings_prices.csv")

In [3]:
reviews_by_user = {}

# review with lower point get less weight in sampling reference

In [4]:
# go through df map to dictionary
# user_id → that user's reviews in date order, each with its row in the df, its day and its rating.
users = recipe_user_interaction_df["user_id"].to_numpy()
days = pd.to_datetime(recipe_user_interaction_df["date"]).to_numpy().astype("datetime64[D]").astype(np.int64)
ratings = recipe_user_interaction_df["rating"].to_numpy()
for row in np.lexsort((days, users)):
    reviews_by_user.setdefault(users[row], []).append({"row": row, "day": days[row], "rating": int(ratings[row])})
print(f"{len(reviews_by_user):,} users, {len(recipe_user_interaction_df):,} reviews")

126,032 users, 512,614 reviews


In [5]:
nutrients = recipe_user_interaction_df[NUTRIENT_COLUMNS].to_numpy(np.float64)
preferences = np.full(nutrients.shape, np.nan)

for user_id, reviews in reviews_by_user.items():
    weight_sum, weighted_sum = 0.0, np.zeros(len(NUTRIENT_COLUMNS))
    for review in reviews:
        weight = 1/pow(2,5-review.get("rating",0))  # 5 → 1, 4 → 1/2, 3 → 1/4, 2 → 1/8, 1 → 1/16, 0 → 1/32
        weight_sum += weight
        weighted_sum += weight * nutrients[review["row"]]
    for review in reviews:
        preferences[review["row"]] = weighted_sum / weight_sum  # Weighted average.

# add column for preference that match ipynb training in UserEncoder for n_prefererence_nutrient
# Rounded to the nearest integer, a half going up (np.round would send it to the even neighbour).
recipe_user_interaction_df[PREFERENCE_COLUMNS] = np.floor(preferences + 0.5)

In [6]:
display(recipe_user_interaction_df[["user_id", "date", "rating", "calories (g)", "preferred calories (g)",
                                    "sugar (g)", "preferred sugar (g)"]].head())

,user_id,date,rating,calories (g),preferred calories (g),sugar (g),preferred sugar (g)
0,4470,2006-02-18,5,103000.0,707016.0,650.0,1382.0
1,593927,2010-08-21,5,103000.0,617198.0,650.0,3280.0
2,178427,2011-12-05,5,103000.0,925498.0,650.0,2720.0
3,319943,2009-07-18,4,346800.0,705501.0,0.0,1226.0
4,674022,2011-04-10,5,346800.0,790817.0,0.0,1718.0


In [7]:
recipe_user_interaction_df.to_csv(INPUT_DIR+"/recipe_user_ratings_preferences_prices.csv", index=False)
print(f"Saved {INPUT_DIR}/recipe_user_ratings_preferences_prices.csv: {len(recipe_user_interaction_df):,} rows, "
      f"{len(PREFERENCE_COLUMNS)} preference columns added")

Saved ./input_stage2/recipe_user_ratings_preferences_prices.csv: 512,614 rows, 7 preference columns added
